In [30]:
import os 
import json

In [31]:
# define functions for dividing the talks into chunks. 
# We want chunks to be large enough to have a good topic but not be so large they are multi topic.
def is_header(text):
    return (
        len(text.split()) < 8
        and text.istitle()
        and not text.endswith(".")
    )

def chunk_talk(body, max_paragraphs=3):
    chunks = []
    current = []

    for line in body:
        line = line.strip()
        if not line:
            continue

        # stop at notes section
        if line.lower() == "notes":
            break

        # if it's a header then start new chunk
        if is_header(line):
            if current:
                chunks.append(" ".join(current))
                current = []
            current.append(line)
            continue

        current.append(line)

        # if chunk is big enough then flush
        if len(current) >= max_paragraphs:
            chunks.append(" ".join(current))
            current = []

    if current:
        chunks.append(" ".join(current))

    return chunks

In [32]:
# Load talks from the folder path

def load_talks(folder_path):
    import os, json

    talks = []

    for root, _, files in os.walk(folder_path):
        session = os.path.basename(root)

        for file in files:
            if not file.endswith(".json"):
                continue

            path = os.path.join(root, file)

            try:
                with open(path, "r", encoding="utf-8") as f:
                    talk = json.load(f)

                if talk is None:
                    print("NULL JSON:", path)
                    continue

                if not isinstance(talk, dict):
                    print("INVALID TYPE:", path, type(talk))
                    continue

                talk["session"] = session
                talks.append(talk)

            except Exception as e:
                print("FAILED:", path, e)

    return talks

In [33]:
def build_chunk_dataset(folder_path):
    talks = load_talks(folder_path)

    all_chunks = []

    for talk in talks:
        body = talk.get("body", [])
        chunks = chunk_talk(body)

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "text": chunk,
                "speaker": talk.get("speaker"),
                "title": talk.get("title"),
                "chunk_id": i,
                "source": talk.get("link")
            })

    return all_chunks

In [34]:
folder = "../data/"  # change this to your path

talk_chunks = build_chunk_dataset(folder)

print(f"Total chunks: {len(talk_chunks)}")
print(talk_chunks[90])

Total chunks: 60927
{'text': 'The great Isaiah foresaw such moments and foretold this very setting in which we find ourselves: “And it shall come to pass in the last days, that the mountain of the Lord’s house shall be established in the top of the mountains, and shall be exalted above the hills; and all nations shall flow unto it. “And many people shall go and say, Come ye, and let us go up to the mountain of the Lord, to the house of the God of Jacob; and he will teach us of his ways, and we will walk in his paths: for out of Zion shall go forth the law, and the word of the Lord from Jerusalem.”', 'speaker': 'Elder Jeffrey R. Holland', 'title': '“The Peaceable Things of the Kingdom”', 'chunk_id': 2, 'source': 'https://www.churchofjesuschrist.org/study/general-conference/1996/10/the-peaceable-things-of-the-kingdom'}


In [37]:
import textwrap
chunk = talk_chunks[-99]


print("TITLE:", chunk["title"])
print("SPEAKER:", chunk["speaker"])
print("-" * 50)
print(textwrap.fill(chunk["text"], width=100))

TITLE: A Safe Place for Marriages and Families
SPEAKER: Barbara B. Smith
--------------------------------------------------
Each child has the love and interest of both mother and father. When children are treated fairly,
there is no cause for jealousy because there is no partiality. Reading the Book of Mormon, we find
that whenever the people were truly committed to the Lord and had the Holy Ghost in their midst, the
conditions described were similar. We read of such an example in 4 Nephi when “every man did deal
justly one with another. “And they had all things common among them; therefore there were not rich
and poor, … but they were all … partakers of the heavenly gift” of love. (4 Ne. 1:2–3.)
